[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/kerr/kerr.ipynb)

# Light past a rotating black hole

Light rays past a rotating black hole: first the same rays as its spin sweeps between opposite directions, then a camera looking at its glowing disk through a rotating polarizer. In Kerr–Schild form the geometry is the flat one plus a null dyad, so the inverse metric is a single map, `Vector + 2 * profile * null * (null | Vector)`. The rays follow its Hamiltonian, and minus its gradient, a vector built from the profile's gradient and the null direction's derivative, is the force on the ray, with no component equations.

The field is the Cartesian Kerr–Schild form of [Pelle et al., *Skylight*, equations 24–26](https://academic.oup.com/mnras/article/515/1/1316/6631564); [Francis and Kosowsky, §VII](https://arxiv.org/pdf/gr-qc/0311007) develop its geometric-algebra construction.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.relativity.kerr import render
from examples.relativity.kerr.core import Field, camera_rays, camera_screen, camera_trace, resolve, trace

In [ ]:
ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Spin = ga.gatype.from_blades("xy xz yz")
Screen = ga.gatype.from_blades("x y")
ScreenFrame = ga.gatype((Vector, Screen))
Coherency = ga.gatype((Screen, Screen))
VectorGradient = ga.gatype((Vector, Vector))
spatial_identity = mv.t | (mv.t ^ Vector)

## 1. The geometry: the identity plus a null dyad

At every event the geometry is a scalar profile and a null direction, together with their derivatives. A scalar's gradient is a vector, whose inner product with a small step is the scalar's change; the null direction's derivative is a map with the step left open, `null_gradient`, and its curl a bivector. The inverse metric is the identity plus twice the profile times the null dyad, and the local frame the identity plus half of it; the gradient of the null direction's overlap with a fixed vector is that vector pulled back through the derivative, by its adjoint. The null direction squares to zero, so the dyad does too, and the metric is the identity minus the same dyad, with nothing to invert.

In [ ]:
def field(position: Vector, spin: Spin, mass: float) -> Field:
    """The Kerr profile, null direction and their derivatives at a batch of events."""
    spatial = mv.t | (mv.t ^ position)                                                      # [...] Vector
    normal = spin * mv.xyz                                                                  # [...] Vector
    projection = normal | spatial                                                           # [...] Scalar
    spin_squared = -spin.squared()                                                          # [...] Scalar
    offset = -spatial.squared() - spin_squared                                              # [...] Scalar
    discriminant = (offset.squared() + 4 * projection.squared()).square_root()              # [...] Scalar
    radius_squared = (offset + discriminant) / 2                                            # [...] Scalar
    radius = radius_squared.square_root()                                                   # [...] Scalar
    # Its inner product with a small step is how far the step changes the oblate radius.
    radius_gradient = -(radius_squared * spatial - normal * projection) / (radius * discriminant)  # [...] Vector
    direction = (radius * spatial + (spatial | spin) - normal * projection / radius) / (radius_squared + spin_squared)  # [...] Vector
    lever = spatial - 2 * radius * direction + normal * projection / radius_squared          # [...] Vector
    direction_gradient = (
        lever * (radius_gradient | Vector)
        + radius * spatial_identity + (spatial_identity | spin)
        - normal * (normal | Vector) / radius
    ) / (radius_squared + spin_squared)                                                     # [...] Vector <- Vector
    denominator = radius_squared.squared() + projection.squared()                           # [...] Scalar
    profile = mass * radius**3 / denominator                                                # [...] Scalar
    profile_gradient = profile * (3 * radius_gradient / radius
        - (4 * radius**3 * radius_gradient + 2 * projection * normal) / denominator)       # [...] Vector
    # The gradient's curl, `(Vector ^ null_gradient(Vector)).contract(1, 2)`, piece by piece: the
    # rank-one piece gives its two vectors wedged, the turn by the spin twice the spin, the rest none.
    null_curl = -((radius_gradient ^ lever) + 2 * spin) / (radius_squared + spin_squared)    # [...] Bivector
    return Field(radius, profile, mv.t - direction, profile_gradient, -direction_gradient, null_curl)  # [...] Field


def metric(geometry: Field) -> VectorGradient:
    """The identity minus the null dyad: a vector to its covector."""
    return Vector - 2 * geometry.profile * geometry.null * (geometry.null | Vector)         # [...] Vector <- Vector


def inverse_metric(geometry: Field) -> VectorGradient:
    """The identity plus the null dyad: covector momentum to the ray's tangent."""
    return Vector + 2 * geometry.profile * geometry.null * (geometry.null | Vector)         # [...] Vector <- Vector


def frame(geometry: Field) -> VectorGradient:
    """The identity plus half the dyad: covector momentum to its direction in the local frame."""
    return Vector + geometry.profile * geometry.null * (geometry.null | Vector)             # [...] Vector <- Vector


def overlap_gradient(geometry: Field, vector: Vector) -> Vector:
    """The gradient of the null direction's overlap with a fixed vector, `null | vector`: the vector
    pulled back through the null direction's derivative, whose changes are spatial."""
    return geometry.null_gradient.adjoint()(mv.t | (mv.t ^ vector))                         # [...] Vector


# A few events around a hole spinning in a tilted plane.
events = mv.vector([[0, 4, 2, 1], [3, -3, 1, 2], [-1, 2, -4, -1]])              # [events] Vector
tilted_spin = 0.7 * mv.xy + 0.2 * mv.yz                                         # [] Spin
geometry = field(events, tilted_spin, 1.0)                                      # [events] Field
null_squared = geometry.null.squared()                                          # [events] Scalar
round_trip = metric(geometry)(inverse_metric(geometry))(mv.x) - mv.x            # [events] Vector

In [ ]:
with np.printoptions(precision=3, suppress=True):
    print("oblate radius:          ", geometry.radius.to_array())
    print("profile:                ", geometry.profile.to_array())
    print("null direction squared: ", null_squared.to_array())
    print("metric of inverse, on x:", round_trip.squared().to_array())

## 2. Rays from two derivatives

A ray's Hamiltonian is half its momentum paired with its image under the inverse metric. Its derivative with respect to momentum is that image: the ray's tangent. Its gradient with respect to position is a vector: the profile's gradient, and the momentum pulled back through the null direction's derivative by its adjoint. Minus that gradient is the rate of the momentum. The momentum is a covector, represented by its Minkowski dual.

In Hamiltonian tensor notation, the ray Hamiltonian is $H=\tfrac12 g^{\mu\nu}p_\mu p_\nu$, and its two derivatives give $\dot x^\mu=\partial H/\partial p_\mu$ and $\dot p_\mu=-\partial H/\partial x^\mu$.

In [ ]:
def motion(geometry: Field, momentum: Vector) -> tuple[Vector, Vector]:
    """The two Hamiltonian derivatives at a field: the ray tangent and minus its position gradient."""
    overlap = geometry.null | momentum                                                      # [...] Scalar
    # Half the momentum paired with its image under the inverse metric, the identity plus the null
    # dyad, is the Hamiltonian; its momentum derivative is that image.
    tangent = momentum + 2 * geometry.profile * geometry.null * overlap                     # [...] Vector
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (geometry.profile_gradient * overlap.squared()
                + 2 * geometry.profile * overlap * overlap_gradient(geometry, momentum))      # [...] Vector
    return tangent, -gradient                                                               # [...] Vector, [...] Vector


def rates(position: Vector, momentum: Vector, spin: Spin, mass: float) -> tuple[Vector, Vector]:
    """The two Hamiltonian derivatives: ray tangent and minus its position gradient."""
    return motion(field(position, spin, mass), momentum)                                    # [...] Vector, [...] Vector

## 3. The same light at every spin

The same rays come in along x past a black hole whose spin sweeps from $-0.8M$ to $+0.8M$ and back, fixed along each ray. Spin changes which rays escape and how far they turn. Inside the horizon the integrator shortens its affine step and stops at a surface within; the black disk is the horizon's equatorial section, and the colours follow each ray's initial offset.

In [ ]:
# G = c = M = 1.
mass = 1.0
frame_count = 80
max_spin = 0.8 * mass
spins = -max_spin * np.cos(np.linspace(0, 2 * np.pi, frame_count, endpoint=False))
duration_ms = 80
extent = 12.0 * mass
rays = 31
offsets = np.linspace(-8, 8, rays) * mass
step_size, steps = 0.04 * mass, 1100
stop_fraction = 0.8
ring_samples = 128

spin = mv.xy * spins[:, None]                                    # [frames, 1] Spin
position = -mv.x * extent + mv.y * offsets                        # [rays] Vector
# A null direction in the local frame becomes a null momentum through the inverse frame.
geometry = field(position, spin, mass)                             # [frames, rays] Field
local_null = mv.t + mv.x                                           # [] Vector
momentum = local_null - geometry.profile * geometry.null * (geometry.null | local_null)  # [frames, rays] Vector
momentum = momentum / (momentum | mv.t)                            # [frames, rays] Vector
position = position.broadcast_to(momentum.shape)                   # [frames, rays] Vector
paths = stack(tuple(event for event, _ in trace(
    position, momentum, spin, mass, step_size, steps, stop_fraction, rates)))   # [steps + 1, frames, rays] Vector

# The horizon's oblate radius gives this equatorial circle in the drawing coordinates.
horizon = mass + np.sqrt(mass**2 - spins**2)                       # [frames]
radius = np.sqrt(horizon**2 + spins**2)                            # [frames]
angles = np.linspace(0, 2 * np.pi, ring_samples + 1)
turns = (mv.xy * (angles / 2)).exp()                               # [ring_samples + 1] Rotor
horizons = (turns >> mv.x) * radius[:, None]                       # [frames, ring_samples + 1] Vector
render.inline(render.animate_rays(paths, horizons, extent), duration_ms)

## 4. Through rotating polarized sunglasses

A camera looks through a rotating linear polarizer at a disk with a prescribed azimuthal magnetic direction. The emitted light is partly polarized perpendicular to that direction as seen across each ray. Parallel transport carries the camera's two screen axes along each ray: the connection gives the plane the local frame turns or boosts in along the ray, and its commutator turns both axes in one expression. The disk emission becomes a `Screen <- Screen` coherency extensor, and pairing a unit analyzer with its image gives the power it passes.

In [ ]:
def connection(geometry: Field, direction: Vector) -> Bivector:
    """The plane the local Kerr–Schild frame turns or boosts in, per unit step along a direction."""
    along = geometry.null | direction                                                       # [...] Scalar
    # The gradient of profile * along.
    turning = geometry.profile_gradient * along + geometry.profile * overlap_gradient(geometry, direction)  # [...] Vector
    return (turning ^ geometry.null) + geometry.profile * geometry.null_curl * along        # [...] Bivector


def polarized_rates(position: Vector, momentum: Vector, screen: ScreenFrame,
                    spin: Spin, mass: float) -> tuple[Vector, Vector, ScreenFrame]:
    """Hamiltonian ray motion and parallel transport of its camera screen, per unit of decreasing
    coordinate time."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    tangent, momentum_rate = motion(geometry, momentum)                                     # [...] Vector, [...] Vector
    # The frame, the identity plus half the dyad, takes covector momentum to its local null direction.
    local_momentum = momentum + geometry.profile * geometry.null * (geometry.null | momentum)  # [...] Vector
    screen_rate = -connection(geometry, local_momentum).commutator(screen)                  # [...] Vector <- Screen
    clock = -(mv.t | tangent)                                                               # [...] Scalar
    return tangent / clock, momentum_rate / clock, screen_rate / clock                      # [...] Vector, [...] Vector, [...] Vector <- Screen


def disk_coherency(position: Vector, momentum: Vector, screen: ScreenFrame,
                  normal: Vector, spin: Spin, mass: float, polarization_degree: float) -> Coherency:
    """Unit-intensity disk emission resolved along the parallel-transported camera axes."""
    local_momentum = frame(field(position, spin, mass))(momentum)                            # [...] Vector
    direction = (mv.t | (mv.t ^ local_momentum)) / (mv.t | local_momentum)                  # [...] Vector
    magnetic = position | (normal * mv.xyz)                                                 # [...] Vector
    magnetic = magnetic / (-magnetic.squared()).square_root()                               # [...] Vector
    # Emission is polarized perpendicular to the magnetic direction projected across the ray.
    electric = (direction ^ magnetic) * mv.xyz                                              # [...] Vector
    polarization = screen.adjoint()(electric)                                               # [...] Screen
    # Polarization vanishes smoothly when looking along the magnetic direction.
    polarized_fraction = -polarization_degree * polarization.squared()                      # [...] Scalar
    return ((1 - polarized_fraction) / 2 * Screen
            - polarization_degree * polarization * (polarization | Screen))                 # [...] Screen <- Screen


def transmitted(coherency: Coherency, analyzers: Screen) -> Scalar:
    """The power each unit analyzer passes: its quadratic readout of the coherency."""
    return -(analyzers | coherency(analyzers))                                              # [...] Scalar


def frequency_ratio(position: Vector, momentum: Vector, camera: Vector, orbit: Bivector,
                    spin: Spin, mass: float) -> Scalar:
    """The frequency a camera at rest at `camera` sees, over the one emitted by gas on a circular
    orbit in the unit plane `orbit`, for rays whose momenta are given at the gas."""
    geometry = field(position, spin, mass)                                                  # [...] Field
    # Kepler's angular speed of a circular equatorial orbit, the spin counted along the orbit.
    alignment = -spin.scalar_product(orbit)                                                 # [] Scalar
    angular_speed = np.sqrt(mass) / (geometry.radius * geometry.radius.square_root() + alignment * np.sqrt(mass))  # [...] Scalar
    flow = mv.t + angular_speed * orbit.commutator(mv.t | (mv.t ^ position))                # [...] Vector
    gas = flow / (flow | metric(geometry)(flow)).square_root()                               # [...] Vector
    at_rest = mv.t / (mv.t | metric(field(camera, spin, mass))(mv.t)).square_root()          # [] Vector
    # The momentum paired with time is conserved along the ray, so the camera's frequency reads here.
    return (momentum | at_rest) / (momentum | gas)                                          # [...] Scalar


def disk_temperature(radius: Scalar, inner_radius: float, hottest: float) -> Scalar:
    """A thin disk's temperature at each oblate radius: zero at its inner edge, `hottest` at 49/36 of
    that radius, and falling as the radius to the power -3/4 beyond."""
    ratio = inner_radius / radius                                                           # [...] Scalar
    root = ratio.square_root()                                                              # [...] Scalar
    # ratio^(3/4) (1 - ratio^(1/2))^(1/4), divided by its value at the hottest radius.
    profile = (ratio * root).square_root() * (1 - root).square_root().square_root()         # [...] Scalar
    return profile * (hottest / ((36 / 49) ** 0.75 * (1 / 7) ** 0.25))                      # [...] Scalar

Each pixel's ray is followed back with its screen transported until it is captured, escapes to the sky, or crosses the disk between its radii; `resolve` keeps where each one ended. Without the polarizer the camera sees the disk lensed around the shadow, its far side bent up over the top.

The gas orbits with the hole, and each disk pixel's frequency ratio is two pairings with the ray's momentum: with the camera at rest, and with the orbiting gas. The disk glows as a blackbody, its outer reaches at about the Sun's temperature, and a blackbody seen at a shifted frequency is a blackbody at the shifted temperature. The side coming towards the camera turns bluer and brighter, the side going away redder and dimmer.

In [ ]:
camera_spin = 0.8 * mass
camera_distance = 28.0 * mass
inclination = np.deg2rad(75.0)
half_view = np.tan(np.deg2rad(19.0))
image_width = 480
image_height = image_width * 2 // 3
step_fraction, camera_steps = 0.1, 720
inner_radius, outer_radius = 3.4 * mass, 12.0 * mass
hottest = 8000.0                                                                    # K, at the hottest radius
escape_radius = 100.0 * mass
capture_radius = 1.002 * (mass + np.sqrt(mass**2 - camera_spin**2))
sky_width, sky_seed = 2048, 8
sky_height = sky_width // 2

orientation = (mv.zx * (inclination / 2)).exp()                                  # [] Rotor
eye = (orientation >> mv.z) * camera_distance                                    # [] Vector
spin = mv.xy * camera_spin                                                       # [] Spin
# Past-directed light from each pixel, and the camera's two screen axes across it.
position, momentum = camera_rays(eye, orientation, image_width, image_height, half_view, spin, mass)   # [pixels] Vector, [pixels] Vector
screen = camera_screen(position, momentum, orientation, spin, mass)              # [pixels] Vector <- Screen
steps = camera_trace((position, momentum, screen), spin, mass, step_fraction, camera_steps, polarized_rates)  # Step generator
image = resolve(steps, image_width * image_height, mv.z, inner_radius, outer_radius,
                escape_radius, capture_radius, spin, mass)                       # Image
points, momenta, screens = image.disk                                            # [disk pixels] Vector, Vector, Vector <- Screen
# The gas orbits with the hole, in its equatorial plane.
shift = frequency_ratio(points, momenta, eye, mv.xy, spin, mass)                     # [disk pixels] Scalar
temperature = shift * disk_temperature(image.disk_radii, inner_radius, hottest)      # [disk pixels] Scalar
sky = render.star_texture(sky_width, sky_height, sky_seed)
render.draw_camera(image, temperature, hottest, (image_height, image_width), sky, inner_radius, outer_radius);

In [ ]:
polarizer_frames = 80
polarizer_angles = np.linspace(0, np.pi, polarizer_frames, endpoint=False)
polarization_degree = 0.8

# The emission where each ray met the disk, read through each analyzer.
coherency = disk_coherency(points, momenta, screens, mv.z, spin, mass, polarization_degree)   # [disk pixels] Screen <- Screen
analyzers = (mv.xy * (polarizer_angles / 2)).exp() >> mv.x                                   # [analyzers] Screen
transmission = transmitted(coherency, analyzers[:, None])                                    # [analyzers, disk pixels] Scalar
render.inline(render.animate_polarizer(image, temperature, hottest, transmission, (image_height, image_width), sky,
                                       inner_radius, outer_radius), duration_ms)

Through the polarizer the exposure is doubled, making up for the half of unpolarized light it stops, so the starlight and unpolarized emission look as in the still; the polarized fraction falls smoothly to zero when looking along the magnetic direction. The disk's orbital texture is illustrative, and its colours are exaggerated in saturation to show the shift.

[Francis and Kosowsky, §II](https://arxiv.org/pdf/gr-qc/0311007) develop the bivector connection; [Gelles et al.](https://arxiv.org/abs/2105.09440) describe polarized Kerr images with prescribed emission.